In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import sqlite3

# 1. Load the dataset
df = pd.read_csv("Buy_Now_Pay_Later_BNPL_CreditRisk_Dataset.csv")

# 2. Create an in-memory SQLite database and load the data
conn = sqlite3.connect(':memory:')
df.to_sql('bnpl_data', conn, index=False)

# 3. SQL Extraction
# We drop existing 'risk_score' and 'customer_segment' because we want to build our own.
# We also drop 'user_id' and 'transaction_date'.
query = """
SELECT
    age, employment_type, monthly_income, credit_score,
    purchase_amount, product_category, bnpl_installments,
    repayment_delay_days, missed_payments, app_usage_frequency,
    location, debt_to_income_ratio, default_flag
FROM
    bnpl_data
"""
cleaned_df = pd.read_sql_query(query, conn)

# 4. One-Hot Encode categorical variables
categorical_cols = ['employment_type', 'product_category', 'location']
cleaned_df = pd.get_dummies(cleaned_df, columns=categorical_cols, drop_first=True)

# 5. Separate features (X) and target (y)
# IMPORTANT: We drop 'repayment_delay_days' and 'missed_payments' to prevent data leakage at checkout
X = cleaned_df.drop(['default_flag', 'repayment_delay_days', 'missed_payments'], axis=1)
y = cleaned_df['default_flag']

# 6. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. Train the Random Forest Model
# Added class_weight='balanced' AND max_depth=10 to prevent overfitting and boost recall
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# 8. Evaluate the new model
y_pred = rf_model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

# 9. Extract updated feature importances
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print("Top 5 Drivers of Default (Pre-Checkout):\n", importances.sort_values(ascending=False).head(5))

Model Accuracy: 60.13%

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.41      0.55      1233
           1       0.50      0.88      0.64       836

    accuracy                           0.60      2069
   macro avg       0.67      0.65      0.60      2069
weighted avg       0.70      0.60      0.59      2069

Top 5 Drivers of Default (Pre-Checkout):
 credit_score            0.365620
monthly_income          0.173212
debt_to_income_ratio    0.148034
app_usage_frequency     0.079245
age                     0.061489
dtype: float64


In [ ]:
# 1. Get the probability of default using your improved model
default_probabilities = rf_model.predict_proba(X_test)[:, 1]

# 2. Create a new DataFrame for the results
results_df = X_test.copy()
results_df['default_probability'] = default_probabilities
results_df['actual_default'] = y_test

# 3. Define the Business Logic for Risk Segments
# (Adjusted thresholds based on the balanced model)
def assign_risk_segment(prob):
    if prob < 0.40:
        return 'Low Risk'
    elif prob < 0.60:
        return 'Medium Risk'
    else:
        return 'High Risk'

results_df['custom_risk_segment'] = results_df['default_probability'].apply(assign_risk_segment)

# 4. Export the data for Power BI
results_df.to_csv('BNPL_PowerBI_Dashboard_Data.csv', index=False)
print("Export complete. File 'BNPL_PowerBI_Dashboard_Data.csv' is ready to download.")

Export complete. File 'BNPL_PowerBI_Dashboard_Data.csv' is ready to download.
